# FaceGuard AI — training notebook template

Save a copy as `notebooks/<task>_<model>.ipynb` (e.g. `antispoofing_mobilenetv2.ipynb`) and fill in each section.
Keep the section order so every notebook reads the same in the report and at the interview.

**Owner:** _name_ · **Task:** _recognition / antispoofing / emotion_ · **Model:** _name_

**References:** _papers or repos you used as a reference (cite them in the report too)_

## 1. Setup
Clones the repo, installs requirements and mounts Drive. Works on Colab, Kaggle and locally.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/henryhoang207/Facial-Recognition---COS30082.git"
BRANCH = "main"  # change to your feature branch if needed

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB or IN_KAGGLE:
    REPO_DIR = Path("/content/repo" if IN_COLAB else "/kaggle/working/repo")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    # torch / torchvision are preinstalled; install the rest only
    reqs = [l for l in (REPO_DIR / "requirements.txt").read_text().splitlines()
            if l and not l.startswith("#") and not l.startswith("torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO_DIR))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/FaceGuard")
elif IN_KAGGLE:
    DRIVE_DIR = Path("/kaggle/working/FaceGuard")  # add data as a Kaggle dataset; download checkpoints at the end
else:
    DRIVE_DIR = REPO_DIR

DATA_DIR = DRIVE_DIR / "data"
CKPT_DIR = DRIVE_DIR / "checkpoints"
FIG_DIR = REPO_DIR / "reports" / "figures"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR, "| data:", DATA_DIR, "| checkpoints:", CKPT_DIR)

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## 2. Config
Every number you might change lives here, so runs are easy to compare.

In [ ]:
CONFIG = {
    "task": "antispoofing",        # recognition | antispoofing | emotion
    "model": "mobilenetv2",    # short name, used in file names
    "img_size": 112,           # must match src.common.face_crop
    "batch_size": 64,
    "epochs": 20,
    "lr": 1e-3,
    "weight_decay": 1e-4,
}
CONFIG

## 3. Data
Load with the shared loaders from `src/common/`. Show a few samples and the class counts.

In [ ]:
# from src.common.face_crop import crop_face
# train_loader, val_loader, test_loader = ...

## 4. Model
Define the model and say why you chose it (backbone, pretrained weights or from scratch, loss).

In [ ]:
# model = ...
# model = model.to(DEVICE)
# print(sum(p.numel() for p in model.parameters() if p.requires_grad), "trainable parameters")

## 5. Train
Log train and val loss / metric every epoch. Keep the best checkpoint by validation metric.

In [ ]:
from datetime import date

def checkpoint_path() -> Path:
    # naming rule: <task>_<model>_<date>.pt
    return CKPT_DIR / f"{CONFIG['task']}_{CONFIG['model']}_{date.today().isoformat()}.pt"

def save_checkpoint(model, epoch: int, metrics: dict):
    path = checkpoint_path()
    torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": metrics, "config": CONFIG}, path)
    print("saved", path)

history = {"train_loss": [], "val_loss": [], "val_metric": []}
# for epoch in range(CONFIG["epochs"]):
#     ...
#     if is_best: save_checkpoint(model, epoch, {...})

## 6. Evaluate
Use the metrics agreed for your task, on the test set only:

- **Recognition:** ROC curve + AUC on `verification_pairs_val.txt` (shared script from Đức Anh), cosine vs Euclidean
- **Anti-spoofing:** accuracy, APCER, BPCER, ACER; test people not in the training set
- **Emotion:** accuracy, per-class F1, confusion matrix

In [ ]:
import matplotlib.pyplot as plt

def save_fig(name: str):
    # naming rule: <task>_<model>_<what>.png
    path = FIG_DIR / f"{CONFIG['task']}_{CONFIG['model']}_{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("saved", path)

# plt.plot(history["train_loss"], label="train"); plt.plot(history["val_loss"], label="val")
# plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); save_fig("loss")

## 7. Results and notes
One table with the final numbers, then 3–5 lines: what worked, what failed, what you would try next. This feeds your Results and Discussion paragraphs.

| Model | Metric 1 | Metric 2 | Notes |
| --- | --- | --- | --- |
|  |  |  |  |